# 08 – Silver History: DESCRIBE HISTORY, Time Travel, CDF, RESTORE

Bu notebook Delta Lake özelliklerini silver tablolar üzerinde gösterir.

**Reset yapmaz.** 07_silver_merge_edge_cases'in bıraktığı state'i okur.

| Bölüm | İçerik |
|---|---|
| 1 | Guard: batch_log'da 12 SUCCESS var mı |
| 2 | DESCRIBE HISTORY: card detay, diğer tablolar özet, operationMetrics ↔ batch_log |
| 3 | Time travel: card satır sayısı version bazında, client 435 önce / sonra, TIMESTAMP AS OF örneği |
| 4 | CDF: iptal edilen kartlar, batch bazında batch_log kontrolü, landing D karşılaştırması |
| 5 | RESTORE: batch 1'e dön, son MERGE version'ına geri gel |

⚠️ Time travel eski dosyalara bağlı. VACUUM çalıştıysa (retention 7 gün) eski version'lar okunamayabilir. Bu durumda önce 07, sonra 08 çalıştırılır.

## 1. Guard

Amaç: 08 reset yapmadığı için, batch 1 → 3'ün 4 tabloya uygulanmış olması gerekir (4 × 3 = 12 SUCCESS).

Çıktı: "Guard OK" ve batch_log tablosu görünmeli. Hata çıkarsa önce 07 çalıştırılır.

In [0]:
from pyspark.sql import functions as F

EXPECTED_SUCCESS = 12

df_log = spark.table("workspace.silver.batch_log")
success_count = df_log.filter(F.col("status") == "SUCCESS").count()

# 08 reset yapmaz; batch'ler uygulanmamışsa sonraki bölümler yanlış sonuç verir, bu yüzden burada durdur
if success_count != EXPECTED_SUCCESS:
    raise Exception(f"batch_log'da {success_count} SUCCESS var, beklenen {EXPECTED_SUCCESS}. Önce 07_silver_merge_edge_cases çalıştırılmalı.")

print("Guard OK: SUCCESS =", success_count)
display(df_log.orderBy("batch_id", "table_name"))

## 2. DESCRIBE HISTORY

Delta her yazma işlemini bir version olarak saklar. DESCRIBE HISTORY bu version'ları gösterir: version, timestamp, operation, operationMetrics.

Son reset (CREATE OR REPLACE) version'ından sonraki MERGE'ler sırayla batch 1, 2, 3'tür. Version numaraları ezbere yazılmaz, history'den okunur.

Card'da üç operasyonun (I / U / D) hepsi olduğu için detay card'da gösterilir.

Çıktı: 3 satır. Beklenen inserted / updated / deleted: batch 1 = 2 / 7 / 4, batch 2 = 5 / 8 / 2, batch 3 = 1 / 6 / 4.

In [0]:
from pyspark.sql import Window

def get_merge_history(table_name):
    full_name = "workspace.silver." + table_name
    df_history = spark.sql("DESCRIBE HISTORY " + full_name)

    # Son reset = son CREATE OR REPLACE version'ı (reset_silver tabloyu overwrite ile yeniden yazar)
    df_resets = df_history.filter(F.col("operation").startswith("CREATE OR REPLACE"))
    reset_version = df_resets.agg(F.max("version")).collect()[0][0]
    print(table_name, "reset_version =", reset_version)

    # Sadece MERGE'ler; serverless'ın otomatik OPTIMIZE gibi işlemleri dışarıda kalır
    df_merges = df_history.filter((F.col("operation") == "MERGE") & (F.col("version") > reset_version))

    # Reset sonrası 1., 2., 3. MERGE = batch 1, 2, 3
    window_by_version = Window.orderBy("version")
    df_merges = df_merges.withColumn("batch_id", F.row_number().over(window_by_version))

    df_result = df_merges.select(
        F.lit(table_name).alias("table_name"),
        "batch_id",
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics")["numTargetRowsInserted"].cast("int").alias("rows_inserted"),
        F.col("operationMetrics")["numTargetRowsUpdated"].cast("int").alias("rows_updated"),
        F.col("operationMetrics")["numTargetRowsDeleted"].cast("int").alias("rows_deleted"),
        "operationMetrics",
    )
    return df_result

df_card_history = get_merge_history("card")
display(df_card_history.orderBy("version"))

### 2b. Özet: 4 tablo ve batch_log karşılaştırması

Amaç: Her tablo için reset sonrası 3 MERGE version'ı ve Delta'nın operationMetrics sayıları ile batch_log'daki sayıların karşılaştırması.

Çıktı: 12 satır. history_* kolonları Delta'dan, log_* kolonları batch_log'dan gelir. ok kolonunun 12 satırda da true olması beklenir.

In [0]:
CDC_TABLES = ["client", "account", "card", "loan"]

df_all_history = None
for table_name in CDC_TABLES:
    df_one = get_merge_history(table_name).drop("operationMetrics")
    if df_all_history is None:
        df_all_history = df_one
    else:
        df_all_history = df_all_history.unionByName(df_one)

df_history_counts = df_all_history.select(
    "table_name",
    "batch_id",
    "version",
    F.col("rows_inserted").alias("history_inserted"),
    F.col("rows_updated").alias("history_updated"),
    F.col("rows_deleted").alias("history_deleted"),
)

df_log_counts = df_log.select(
    "table_name",
    "batch_id",
    F.col("rows_inserted").alias("log_inserted"),
    F.col("rows_updated").alias("log_updated"),
    F.col("rows_deleted").alias("log_deleted"),
)

# Delta'nın kendi sayıları ile bizim log'umuz aynı MERGE'ü anlatıyor mu
df_compare = df_history_counts.join(df_log_counts, on=["table_name", "batch_id"], how="left")

is_ok = (
    (F.col("history_inserted") == F.col("log_inserted"))
    & (F.col("history_updated") == F.col("log_updated"))
    & (F.col("history_deleted") == F.col("log_deleted"))
)
df_compare = df_compare.withColumn("ok", is_ok)

display(df_compare.orderBy("table_name", "batch_id"))
print("ok satır sayısı:", df_compare.filter(F.col("ok")).count(), "/ 12")

## 3. Time travel

Delta eski version'ları saklar. VERSION AS OF ile tablo geçmişteki bir haliyle okunabilir.

Timestamp yerine version kullanılıyor: runner batch'leri birkaç saniye arayla çalıştırdığı için timestamp kırılgan olur.

### 3a. Card: version başına satır sayısı

Version'lar history'den okunur. Batch 1'den hemen önceki version reset sonrası state'tir (after_batch = 0).

Çıktı: 4 satır. Beklenen row_count: 892 → 890 → 893 → 890.

In [0]:
card_merge_rows = df_card_history.orderBy("version").select("batch_id", "version").collect()

# Batch 1'den hemen önceki version = reset sonrası state (arada CDF ALTER TABLE olabilir, veri aynı)
first_merge_version = card_merge_rows[0]["version"]
versions_to_check = [(0, first_merge_version - 1)]
for row in card_merge_rows:
    versions_to_check.append((row["batch_id"], row["version"]))

results = []
for batch_id, version in versions_to_check:
    query = f"SELECT COUNT(*) AS row_count FROM workspace.silver.card VERSION AS OF {version}"
    row_count = spark.sql(query).collect()[0]["row_count"]
    results.append((batch_id, version, row_count))

df_card_versions = spark.createDataFrame(results, ["after_batch", "version", "row_count"])
display(df_card_versions.orderBy("version"))

### 3b. Client 435: batch 2 öncesi ve sonrası

Batch 2'de client 435 iki kez taşındı (U → U): önce district 32, sonra 28. Dedup son kaydı aldı.

Batch 2 öncesi = batch 2 MERGE version'ı - 1. Batch 1'de bu müşteri değişmediği için öncesi bronze'daki değerdir.

Çıktı: before satırında district_id bronze ile aynı ve _change_ts 1998-12-31; after satırında district_id 28 ve _change_ts 1999-02-18. Ara değer 32 hiçbir version'da yok.

In [0]:
df_client_history = get_merge_history("client")
batch2_version = df_client_history.filter(F.col("batch_id") == 2).collect()[0]["version"]
before_version = batch2_version - 1

query_before = f"SELECT client_id, district_id, _change_ts FROM workspace.silver.client VERSION AS OF {before_version} WHERE client_id = 435"
query_after = f"SELECT client_id, district_id, _change_ts FROM workspace.silver.client VERSION AS OF {batch2_version} WHERE client_id = 435"

df_before = spark.sql(query_before).withColumn("state", F.lit("before batch 2")).withColumn("version", F.lit(before_version))
df_after = spark.sql(query_after).withColumn("state", F.lit("after batch 2")).withColumn("version", F.lit(batch2_version))

# Bronze'da her kolon string; karşılaştırma için sadece print
df_bronze_435 = spark.table("workspace.bronze.client").filter(F.col("client_id") == "435")
bronze_district = df_bronze_435.collect()[0]["district_id"]
print("bronze district_id:", bronze_district)

display(df_before.unionByName(df_after))

### 3c. TIMESTAMP AS OF (tek örnek)

Time travel timestamp ile de yapılabilir: o andan önceki son commit'in version'ı okunur.

Timestamp history'den okunur (card batch 2 MERGE'ü). Bu projede ana yöntem VERSION AS OF; bu hücre sadece "bu da var" demek için.

Çıktı: row_count 893 (version 20 ile aynı).

In [0]:
df_card_batch2 = df_card_history.filter(F.col("batch_id") == 2)

# Timestamp Spark içinde string'e çevrilir; Python datetime'a çekince timezone kayabilir
df_card_batch2 = df_card_batch2.withColumn("ts_text", F.date_format("timestamp", "yyyy-MM-dd HH:mm:ss.SSS"))
batch2_row = df_card_batch2.select("version", "ts_text").collect()[0]

query = f"SELECT COUNT(*) AS row_count FROM workspace.silver.card TIMESTAMP AS OF '{batch2_row['ts_text']}'"
row_count = spark.sql(query).collect()[0]["row_count"]

print("batch 2 version  :", batch2_row["version"])
print("batch 2 timestamp:", batch2_row["ts_text"])
print("row_count        :", row_count)

## 4. Change Data Feed (CDF)

CDF açık bir tabloda Delta her değişikliği satır seviyesinde kaydeder. table_changes() ile okunur.

_change_type: insert, update_preimage (önceki hal), update_postimage (sonraki hal), delete.

_commit_version hangi version'da (yani hangi batch'te) değiştiğini gösterir.

Başlangıç version'ı = reset sonrası ilk MERGE version'ı (history'den okunur).

### 4a. İptal edilen kartlar

Silver'da kart iptali hard delete. Silinen kartlar silver'da artık yok ama CDF'te son halleriyle duruyor.

Çıktı: _change_type = delete olan kartlar. Beklenen toplam 10 (batch 1 = 4, batch 2 = 2, batch 3 = 4).

In [0]:
# Reset sonrası ilk MERGE; reset ve CDF ALTER TABLE gibi ara version'lar hesaba girmez
start_version = df_card_history.agg(F.min("version")).collect()[0][0]
print("CDF start_version:", start_version)

# Bitiş = son MERGE; sonradan gelen RESTORE version'ları CDF sayılarına karışmasın
end_version = df_card_history.agg(F.max("version")).collect()[0][0]
print("CDF end_version:", end_version)

query = f"SELECT * FROM table_changes('workspace.silver.card', {start_version}, {end_version})"
df_card_changes = spark.sql(query)

# _commit_version → batch_id eşlemesi history'den
df_version_to_batch = df_card_history.select(F.col("version").alias("_commit_version"), "batch_id")
df_card_changes = df_card_changes.join(df_version_to_batch, on="_commit_version", how="left")

df_cancelled_cards = df_card_changes.filter(F.col("_change_type") == "delete")

display(
    df_cancelled_cards
    .select("batch_id", "_commit_version", "card_id", "disp_id", "type", "issued_date", "_commit_timestamp")
    .orderBy("batch_id", "card_id")
)
print("İptal edilen kart sayısı:", df_cancelled_cards.count())

### 4b. CDF ↔ batch_log (card, batch bazında)

Amaç: CDF'teki satır seviyesindeki değişiklik sayıları ile batch_log'daki sayıların aynı olduğunu göstermek.

Her update CDF'te iki satır üretir (update_preimage + update_postimage); sadece update_postimage sayılır.

Çıktı: 3 satır, ok kolonu 3'ünde de true. Beklenen ins / upd / del: batch 1 = 2 / 7 / 4, batch 2 = 5 / 8 / 2, batch 3 = 1 / 6 / 4.

In [0]:
change_types = ["insert", "update_postimage", "delete"]

# _change_type değerleri kolonlara açılır; batch'te olmayan tip NULL gelir, 0 yapılır
df_cdf_pivot = df_card_changes.groupBy("batch_id").pivot("_change_type", change_types).count()

df_cdf_counts = df_cdf_pivot.select(
    "batch_id",
    F.coalesce(F.col("insert"), F.lit(0)).alias("cdf_inserted"),
    F.coalesce(F.col("update_postimage"), F.lit(0)).alias("cdf_updated"),
    F.coalesce(F.col("delete"), F.lit(0)).alias("cdf_deleted"),
)

df_card_log_counts = df_log_counts.filter(F.col("table_name") == "card").drop("table_name")

df_cdf_compare = df_cdf_counts.join(df_card_log_counts, on="batch_id", how="left")

is_cdf_ok = (
    (F.col("cdf_inserted") == F.col("log_inserted"))
    & (F.col("cdf_updated") == F.col("log_updated"))
    & (F.col("cdf_deleted") == F.col("log_deleted"))
)
df_cdf_compare = df_cdf_compare.withColumn("ok", is_cdf_ok)

display(df_cdf_compare.orderBy("batch_id"))

### 4c. Landing ↔ CDF + quarantine

Landing'deki her card kaydının silver'a nasıl yansıdığı. İki adım sayıları değiştirir:

- Quarantine: target'ta olmayan key'e gelen U / D (card 101254, batch 3 D).

- Dedup: aynı batch'te I'sı olan key'in U'su tek bir insert'e birleşir (card 1253, batch 2 I → U).

Formül: landing = cdf + quarantine + merged_by_dedup

Çıktı: 3 satır (I, U, D), ok kolonu 3'ünde de true. Beklenen U: 22 = 21 + 0 + 1; D: 11 = 10 + 1 + 0.

In [0]:
# Landing card batch'leri (schema inference yeterli; sadece sayıyoruz)
df_landing_card = spark.read.json("/Volumes/workspace/bronze/landing/card/batch_*/")

landing_i = df_landing_card.filter(F.col("op_type") == "I").count()
landing_u = df_landing_card.filter(F.col("op_type") == "U").count()
landing_d = df_landing_card.filter(F.col("op_type") == "D").count()

# Aynı batch'te aynı key'in I'sı da varsa U, dedup'ta I'ya birleşir (I → U kuralı)
df_landing_inserts = df_landing_card.filter(F.col("op_type") == "I").select("card_id", "batch_id")
df_landing_updates = df_landing_card.filter(F.col("op_type") == "U").select("card_id", "batch_id")
df_u_merged = df_landing_updates.join(df_landing_inserts, on=["card_id", "batch_id"], how="inner")
u_merged_by_dedup = df_u_merged.count()
print("Dedup'ta I'ya birleşen U kayıtları:", df_u_merged.collect())

df_card_quarantine = spark.table("workspace.silver.quarantine").filter(F.col("table_name") == "card")
quarantine_u = df_card_quarantine.filter(F.col("op_type") == "U").count()
quarantine_d = df_card_quarantine.filter(F.col("op_type") == "D").count()

cdf_i = df_card_changes.filter(F.col("_change_type") == "insert").count()
cdf_u = df_card_changes.filter(F.col("_change_type") == "update_postimage").count()
cdf_d = df_card_changes.filter(F.col("_change_type") == "delete").count()

rows = [
    ("I", landing_i, cdf_i, 0, 0),
    ("U", landing_u, cdf_u, quarantine_u, u_merged_by_dedup),
    ("D", landing_d, cdf_d, quarantine_d, 0),
]
df_landing_compare = spark.createDataFrame(rows, ["op_type", "landing", "cdf", "quarantine", "merged_by_dedup"])

expected_landing = F.col("cdf") + F.col("quarantine") + F.col("merged_by_dedup")
df_landing_compare = df_landing_compare.withColumn("ok", F.col("landing") == expected_landing)

display(df_landing_compare)

## 5. RESTORE

RESTORE tabloyu geçmişteki bir version'a geri döndürür. Mülakat sorusu: "Yanlış batch geldi, nasıl geri alırsın?"

Güvenli akış: card önce batch 1 version'ına döner, sonra son MERGE version'ına geri getirilir. Böylece batch_log ile silver sonunda tutarlı kalır.

RESTORE history'yi silmez; yeni bir version olarak eklenir (operation = RESTORE).

Çıktı: batch 1'e dönünce 890 satır ve card 1253 yok (batch 2'de eklendi). Son MERGE'e dönünce 890 satır ve card 1253 gold.

⚠️ Gerçek hayatta RESTORE sonrası batch_log da güncellenmeli. Yoksa geri alınan batch SUCCESS göründüğü için runner onu tekrar çalıştırmaz.

In [0]:
batch1_version = df_card_history.filter(F.col("batch_id") == 1).collect()[0]["version"]
last_merge_version = df_card_history.agg(F.max("version")).collect()[0][0]
print("batch 1 version:", batch1_version, "| son MERGE version:", last_merge_version)

def show_card_state(label):
    df_card = spark.table("workspace.silver.card")
    row_count = df_card.count()
    rows_1253 = df_card.filter(F.col("card_id") == 1253).collect()
    if len(rows_1253) == 0:
        type_1253 = "yok"
    else:
        type_1253 = rows_1253[0]["type"]
    print(label, "| rows:", row_count, "| card 1253:", type_1253)

# 1) Yanlış batch senaryosu: batch 2 ve 3 geri alınır
spark.sql(f"RESTORE TABLE workspace.silver.card TO VERSION AS OF {batch1_version}")
show_card_state("batch 1'e RESTORE sonrası  ")

# 2) Son MERGE haline geri dön; batch_log ile silver yine tutarlı
spark.sql(f"RESTORE TABLE workspace.silver.card TO VERSION AS OF {last_merge_version}")
show_card_state("son MERGE'e RESTORE sonrası")

df_card_last_ops = spark.sql("DESCRIBE HISTORY workspace.silver.card").select("version", "timestamp", "operation", "operationParameters")
display(df_card_last_ops.orderBy(F.col("version").desc()).limit(3))